# Fase 1 · Definición del problema y entorno reproducible
**Pertenencia al SNSS y nivel de complejidad de los establecimientos de salud en Chile**

**Grupo 1:** Brian Cerda · Kattia Espinoza · Karla Martínez · Christopher Ulloa

Este cuaderno define el estudio, documenta el archivo y prepara un entorno reproducible. La limpieza, los filtros y la descripción de resultados se desarrollarán en F2. La evaluación estadística de asociación se proyecta para F3–F4.

**Recorrido:** definir → verificar → organizar → documentar → validar → guardar.

## Introducción

La red asistencial chilena comprende establecimientos con distintos niveles de complejidad, cuyas capacidades asistenciales son relevantes para la derivación de pacientes. En este ámbito coexisten establecimientos pertenecientes al Sistema Nacional de Servicios de Salud (SNSS) y establecimientos que no forman parte de él. Analizar las diferencias en la distribución de sus niveles de complejidad puede aportar antecedentes relevantes para la planificación de la red asistencial, la coordinación de derivaciones y la asignación de recursos. Este estudio examina dicha relación a partir del registro de establecimientos de salud, considerando una población delimitada según criterios de vigencia y disponibilidad de datos.

La unidad de observación será un establecimiento identificado por su código vigente. No equipararemos el número de establecimientos con capacidad de atención ni con calidad del servicio.

## 1. Definición del problema
Guardaremos la definición en `PROYECTO` para que F2 recupere la misma pregunta y los mismos criterios. Esto evita mantener versiones distintas de los objetivos. Las dos variables se estudiarán como categorías; no se entrenará un modelo predictivo en estas fases.

In [1]:
# Configuración central del estudio. Los valores desconocidos se declaran explícitamente.
PROYECTO = {'titulo': 'Pertenencia al SNSS y nivel de complejidad de los establecimientos de salud en Chile',
 'grupo': 'Grupo 1',
 'integrantes': ['Brian Cerda', 'Kattia Espinoza', 'Karla Martínez', 'Christopher Ulloa'],
 'introduccion': 'La red asistencial chilena comprende establecimientos con distintos niveles de '
                 'complejidad, cuyas capacidades asistenciales son relevantes para la derivación '
                 'de pacientes. En este ámbito coexisten establecimientos pertenecientes al '
                 'Sistema Nacional de Servicios de Salud (SNSS) y establecimientos que no forman '
                 'parte de él. Analizar las diferencias en la distribución de sus niveles de '
                 'complejidad puede aportar antecedentes relevantes para la planificación de la '
                 'red asistencial, la coordinación de derivaciones y la asignación de recursos. '
                 'Este estudio examina dicha relación a partir del registro de establecimientos de '
                 'salud, considerando una población delimitada según criterios de vigencia y '
                 'disponibilidad de datos.',
 'problematica': 'En Chile coexisten establecimientos de salud que pertenecen al SNSS y otros que '
                 'no, con distintos niveles de complejidad. El registro contiene ambas variables. '
                 'Para este estudio, se requiere caracterizar cómo se relacionan y determinar si '
                 'la distribución de los niveles de complejidad difiere según la pertenencia al '
                 'SNSS.',
 'pregunta': '¿Existe asociación entre la pertenencia al SNSS y el nivel de complejidad en los '
             'establecimientos de salud vigentes con código único y con ambas variables informadas '
             'en la base?',
 'objetivo_general': 'Evaluar la asociación entre la pertenencia al SNSS y el nivel de complejidad '
                     'de los establecimientos de salud vigentes con código único y ambas variables '
                     'informadas.',
 'objetivos_especificos': ['Delimitar el problema de investigación, la pregunta, el alcance, las '
                           'restricciones y los supuestos del análisis (Fase 1).',
                           'Definir la población de análisis mediante criterios explícitos y '
                           'reproducibles, documentados en una tabla de filtros (Fase 2).',
                           'Estandarizar las categorías de pertenencia al SNSS y de nivel de '
                           'complejidad, y validar los datos resultantes (Fase 2).',
                           'Describir la distribución de los establecimientos según su pertenencia '
                           'al SNSS y su nivel de complejidad (Fase 2).',
                           'Evaluar la asociación entre ambas variables mediante una tabla de '
                           'contingencia, una prueba de independencia y una medida de magnitud de '
                           'la asociación (Fases 3 y 4, pendientes).'],
 'alcance': 'El estudio considera los establecimientos vigentes del conjunto de datos que cumplen '
            'los criterios de inclusión y examina la relación entre su pertenencia al SNSS y su '
            'nivel de complejidad. Quedan fuera del alcance el análisis territorial, los servicios '
            'de urgencia y las conclusiones causales o sobre la calidad de la atención.',
 'dimensiones': {'principales': [{'dimension': 'Administrativa',
                                   'columna': 'TipoPertenenciaEstabGlosa',
                                   'variable': 'pertenencia_snss',
                                   'categorias': ['SNSS', 'No SNSS']},
                                  {'dimension': 'Asistencial',
                                   'columna': 'NivelComplejidadEstabGlosa',
                                   'variable': 'complejidad',
                                   'categorias': ['Baja', 'Mediana', 'Alta']}],
                 'contexto': {'Geográfica': 'Macrozona; se conserva como contexto y no se incluye en el cruce principal.',
                              'Servicios': 'Se conserva como contexto y no se incluye en el cruce principal.'}},
 'denominador': {'general': 'Establecimientos vigentes con código único, pertenencia aplicable y '
                            'complejidad aplicable y registrada (Baja, Mediana o Alta); no el total '
                            'del archivo.',
                 'porcentajes': 'Por fila: dentro de cada grupo de pertenencia (SNSS y No SNSS). El '
                                'porcentaje por columna se usa solo como apoyo.'},
 'restricciones': ['El archivo es un corte del registro, no una serie temporal. El nombre sugiere '
                   '2026-11-03, pero la fecha de descarga y la fecha exacta de corte no están '
                   'acreditadas en los archivos proporcionados.',
                   'El archivo también contiene establecimientos cerrados; la población vigente se '
                   'delimitó durante la Fase 2 a partir del estado de funcionamiento registrado.',
                   'Las conclusiones se limitan a establecimientos vigentes con código único, '
                   'pertenencia clasificable y complejidad baja, mediana o alta. No se extrapolan '
                   'a los registros excluidos.',
                   'La cantidad de establecimientos no mide camas, capacidad resolutiva efectiva, '
                   'volumen de atenciones ni acceso de la población.'],
 'supuestos': ['La pertenencia al SNSS y el nivel de complejidad están correctamente registrados '
               'en la fuente; se comprobará consistencia interna, no exactitud clínica o '
               'administrativa externa.',
               'Cada código vigente identifica a un único establecimiento; su presencia y unicidad '
               'serán comprobadas.',
               'La vigencia se define por el estado registrado. Se aceptarán los estados de '
               'operación habitual o transitoria si aparecen.',
               'No Aplica representa una categoría explícita y se distingue de una celda vacía. '
               'Ninguna permite asignar una de las categorías analíticas requeridas.'],
 'criterios_exito': ['Los cuadernos se ejecutan en orden desde un kernel reiniciado y con rutas '
                     'relativas.',
                     'Cada exclusión se documenta y las filas iniciales se reconcilian con las '
                     'retenidas y excluidas.',
                     'La población final cumple las reglas de inclusión y tiene códigos únicos.',
                     'Los gráficos y porcentajes usan denominadores explícitos y responden a los '
                     'objetivos de F2.',
                     'La matriz final es numérica, binaria, sin nulos y conserva la '
                     'correspondencia con los establecimientos.'],
 'fuente': {'institucion': 'Ministerio de Salud de Chile, Departamento de Estadísticas e '
                           'Información de Salud (DEIS)',
            'archivo': 'establecimientos_20261103.csv',
            'diccionario': 'Diccionario de Datos Base de Establecimientos de Salud Red Asistencial '
                           '2026',
            'portal_institucional': 'https://deis.minsal.cl/sistemas/',
            'fecha_sugerida_nombre': '2026-11-03',
            'fecha_descarga_original': None,
            'fecha_corte_confirmada': None,
            'nota': 'El portal institucional es una referencia general; no acredita por sí solo la '
                    'versión exacta del archivo utilizado en este estudio.'}}

### Presentación de la pregunta y los objetivos
Mostraremos la definición central y comprobaremos que las claves necesarias están presentes.

In [2]:
def presentar_proyecto(configuracion):
    """Comprueba los campos centrales y presenta el estudio."""
    obligatorios = ["titulo", "pregunta", "objetivo_general", "objetivos_especificos", "dimensiones", "denominador"]
    faltantes = [campo for campo in obligatorios if campo not in configuracion]
    if faltantes:
        raise KeyError(f"Faltan campos del proyecto: {faltantes}")
    for campo in ["titulo", "problematica", "pregunta", "objetivo_general", "alcance"]:
        print(f"{campo.upper()}: {configuracion[campo]}\n")
    for numero, objetivo in enumerate(configuracion["objetivos_especificos"], start=1):
        print(f"{numero}. {objetivo}")
    print("\nDIMENSIONES DEL CRUCE")
    for dimension in configuracion["dimensiones"]["principales"]:
        print(f"- {dimension['dimension']}: {dimension['columna']} → {dimension['variable']} ({' / '.join(dimension['categorias'])})")
    for nombre, uso in configuracion["dimensiones"]["contexto"].items():
        print(f"- {nombre}: {uso}")
    print("\nDENOMINADOR:", configuracion["denominador"]["general"])
    print("PORCENTAJES:", configuracion["denominador"]["porcentajes"])

presentar_proyecto(PROYECTO)

TITULO: Pertenencia al SNSS y nivel de complejidad de los establecimientos de salud en Chile

PROBLEMATICA: En Chile coexisten establecimientos de salud que pertenecen al SNSS y otros que no, con distintos niveles de complejidad. El registro contiene ambas variables. Para este estudio, se requiere caracterizar cómo se relacionan y determinar si la distribución de los niveles de complejidad difiere según la pertenencia al SNSS.

PREGUNTA: ¿Existe asociación entre la pertenencia al SNSS y el nivel de complejidad en los establecimientos de salud vigentes con código único y con ambas variables informadas en la base?

OBJETIVO_GENERAL: Evaluar la asociación entre la pertenencia al SNSS y el nivel de complejidad de los establecimientos de salud vigentes con código único y ambas variables informadas.

ALCANCE: El estudio considera los establecimientos vigentes del conjunto de datos que cumplen los criterios de inclusión y examina la relación entre su pertenencia al SNSS y su nivel de compleji

### Alcance, restricciones y supuestos
Trabajaremos con una población delimitada por vigencia, código único y categorías analíticas válidas. “Ambas variables informadas” significará pertenencia o no pertenencia al SNSS y complejidad baja, mediana o alta. `No Aplica` se conservará en el diagnóstico, pero no se tratará como un cuarto nivel de complejidad.

La pertenencia al SNSS no se sustituirá por la clasificación público/privado. Tampoco confundiremos nivel de atención con nivel de complejidad.

**Dimensiones y denominador.** El cruce usa dos dimensiones: la **administrativa** (`TipoPertenenciaEstabGlosa` → `pertenencia_snss`: SNSS / No SNSS) y la **asistencial** (`NivelComplejidadEstabGlosa` → `complejidad`: Baja / Mediana / Alta). La geográfica (macrozona) y la de servicios quedan como contexto. El denominador general es la población de análisis que se construirá en F2, no el total del archivo. Los porcentajes se calcularán por fila, dentro de cada grupo de pertenencia; el porcentaje por columna solo servirá de apoyo.

In [3]:
for apartado in ["restricciones", "supuestos", "criterios_exito"]:
    print(apartado.upper())
    for elemento in PROYECTO[apartado]:
        print("-", elemento)
    print()

RESTRICCIONES
- El archivo es un corte del registro, no una serie temporal. El nombre sugiere 2026-11-03, pero la fecha de descarga y la fecha exacta de corte no están acreditadas en los archivos proporcionados.
- El archivo también contiene establecimientos cerrados; la población vigente se delimitó durante la Fase 2 a partir del estado de funcionamiento registrado.
- Las conclusiones se limitan a establecimientos vigentes con código único, pertenencia clasificable y complejidad baja, mediana o alta. No se extrapolan a los registros excluidos.
- La cantidad de establecimientos no mide camas, capacidad resolutiva efectiva, volumen de atenciones ni acceso de la población.

SUPUESTOS
- La pertenencia al SNSS y el nivel de complejidad están correctamente registrados en la fuente; se comprobará consistencia interna, no exactitud clínica o administrativa externa.
- Cada código vigente identifica a un único establecimiento; su presencia y unicidad serán comprobadas.
- La vigencia se define p

### Plan de análisis y trazabilidad
| Objetivo | Variables o información | Evidencia | Fase |
|---|---|---|---|
| Delimitar el problema | Pregunta, alcance y supuestos | Configuración y ficha | F1 |
| Definir la población | Código y estado; pertenencia y complejidad | Tabla secuencial de filtros y exclusiones | F2 |
| Estandarizar categorías | Pertenencia y complejidad | Tabla antes/después y validaciones | F2 |
| Describir la distribución | Pertenencia y complejidad | Frecuencias, porcentajes y gráficos | F2 |
| Evaluar asociación | Tabla de contingencia 2 × 3 | Prueba de independencia y V de Cramér, previa revisión de condiciones | F3–F4 |

En F2 describiremos diferencias observadas. La significación estadística no se deducirá de los gráficos. Al trabajar con un registro completo de la población delimitada, en F3–F4 se discutirá qué población o proceso justificaría una inferencia más allá de este corte.

## 2. Verificación del entorno
Comprobaremos el intérprete y las bibliotecas. Usaremos `pandas` para tablas, `numpy` para comprobaciones numéricas y `matplotlib` para gráficos. No se requiere un generador automático de perfiles: el reporte de F2 se construirá con operaciones sencillas y explícitas.

In [4]:
from pathlib import Path
import sys
import json
import platform
import importlib
import subprocess
import shutil
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
SEMILLA = 42
np.random.seed(SEMILLA)
# El flujo principal no tiene muestreo aleatorio; la semilla queda declarada.

def verificar_entorno():
    """Devuelve las versiones utilizadas y el estado del entorno virtual."""
    versiones = {}
    for nombre in ["pandas", "numpy", "matplotlib", "nbformat", "nbclient", "ipykernel"]:
        modulo = importlib.import_module(nombre)
        versiones[nombre] = modulo.__version__
    print("Python:", platform.python_version())
    print("Entorno virtual:", sys.prefix != sys.base_prefix)
    display(pd.DataFrame(list(versiones.items()), columns=["biblioteca", "version"]))
    return versiones

versiones = verificar_entorno()

Python: 3.13.5
Entorno virtual: True


,biblioteca,version
0,pandas,3.0.5
1,numpy,2.5.2
2,matplotlib,3.11.1
3,nbformat,5.11.1
4,nbclient,0.11.0
5,ipykernel,7.4.0


**Interpretación.** Las versiones visibles documentan el entorno que ejecutó el cuaderno. Si el indicador de entorno virtual es falso, la ejecución utilizó un entorno compartido; no se presentará como un entorno aislado. El archivo de dependencias permitirá preparar uno propio.

## 3. Estructura del proyecto y archivos de configuración
Organizaremos los resultados con rutas relativas. `data/raw` conservará el CSV recibido; `data/processed` contendrá las salidas de F2. Las figuras y tablas de decisiones quedarán en `docs`.

Los cuadernos se podrán ejecutar desde la raíz o desde su carpeta de fase. Se generarán archivos derivados del proyecto sin modificar el CSV original.

In [5]:
def localizar_raiz(inicio=None):
    """Devuelve la raíz desde la carpeta del proyecto o desde F1/F2."""
    carpeta = Path.cwd() if inicio is None else Path(inicio)
    return carpeta.parent if carpeta.name in {"F1", "F2"} else carpeta

RAIZ = localizar_raiz()
CARPETAS = ["F1", "F2", "src", "data/raw", "data/processed", "docs", "docs/figuras"]
for carpeta in CARPETAS:
    (RAIZ / carpeta).mkdir(parents=True, exist_ok=True)

dependencias = [f"{nombre}=={version}" for nombre, version in versiones.items()]
(RAIZ / "requirements.txt").write_text("\n".join(dependencias) + "\n", encoding="utf-8")
(RAIZ / ".gitignore").write_text(".venv/\n__pycache__/\n.ipynb_checkpoints/\n.env\n", encoding="utf-8")
display(pd.DataFrame({"carpeta": CARPETAS, "existe": [(RAIZ / c).is_dir() for c in CARPETAS]}))

,carpeta,existe
0,F1,True
1,F2,True
2,src,True
3,data/raw,True
4,data/processed,True
5,docs,True
6,docs/figuras,True


**Interpretación.** Las carpetas existentes separan la fuente de las transformaciones y sus evidencias. El archivo de dependencias registra las versiones realmente utilizadas. Git ignorará el entorno local y las cachés, pero no los cuadernos ni los resultados del estudio.

## 4. Módulo de funciones reutilizables
El módulo `src/utilidades.py` contiene tres funciones breves: localizar la raíz, normalizar texto y cargar el CSV. Lo importaremos para comprobar que las fases podrán compartir esas tareas. La prueba de normalización usa datos de ejemplo; no modifica el dataset en F1.

In [6]:
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from src.utilidades import normalizar_texto, cargar_datos

# Caso normal: espacios y mayúsculas; caso límite: valor ausente.
ejemplo = pd.Series(["  Alta   Complejidad  ", None], dtype="string")
resultado = normalizar_texto(ejemplo)
assert resultado.iloc[0] == "alta complejidad"
assert pd.isna(resultado.iloc[1])

# Excepción esperada: la función debe explicar qué archivo falta.
try:
    cargar_datos(RAIZ / "archivo_inexistente_para_prueba.csv")
except FileNotFoundError:
    print("Pruebas correctas: normalización, faltante y archivo inexistente.")
else:
    raise AssertionError("Se esperaba FileNotFoundError.")

Pruebas correctas: normalización, faltante y archivo inexistente.


**Interpretación.** La normalización conserva los faltantes y corrige diferencias de presentación. La carga detecta un archivo inexistente con una excepción explícita. Estas comprobaciones cubren comportamientos que podrían alterar o interrumpir F2.

## 5. Selección y ficha del conjunto de datos
Verificaremos las dimensiones y las columnas sin limpiar ni filtrar filas. El archivo recibido es `establecimientos_20261103.csv`; su diccionario corresponde al registro de establecimientos de salud de 2026.

**Procedencia:** Ministerio de Salud de Chile, DEIS. Portal institucional: https://deis.minsal.cl/sistemas/ . La referencia acredita el contexto del registro, pero no una descarga exacta de esta versión. La fecha `2026-11-03` se deduce únicamente del nombre; la fecha original de descarga y el corte administrativo exacto siguen sin estar documentados.

**Referencias de los materiales proporcionados:**
- Departamento de Estadísticas e Información de Salud. (s. f.). *Establecimientos de salud* [Archivo CSV proporcionado: establecimientos_20261103.csv]. Ministerio de Salud de Chile.
- Departamento de Estadísticas e Información de Salud. (2026). *Diccionario de Datos Base de Establecimientos de Salud Red Asistencial 2026* [Diccionario de datos proporcionado].

La licencia específica no está acreditada en los archivos entregados; no se atribuirá una licencia sin verificarla.

In [7]:
ARCHIVO = RAIZ / "data/raw" / PROYECTO["fuente"]["archivo"]
datos_verificacion = cargar_datos(ARCHIVO)
huella_original = hashlib.sha256(ARCHIVO.read_bytes()).hexdigest()
ficha = pd.DataFrame({
    "caracteristica": ["Filas", "Columnas", "Unidad", "Separador", "Codificación", "Fecha de descarga"],
    "valor": [len(datos_verificacion), datos_verificacion.shape[1], "Establecimiento con código vigente", ";", "UTF-8", "No documentada"]
})
display(ficha)
print("SHA-256 del archivo conservado:", huella_original)

,caracteristica,valor
0,Filas,5837
1,Columnas,33
2,Unidad,Establecimiento con código vigente
3,Separador,;
4,Codificación,UTF-8
5,Fecha de descarga,No documentada


SHA-256 del archivo conservado: febff55e3612fb60076df9279f1090240cd02a39e9c7b79b822119dfdef84c8d


**Interpretación.** El archivo tiene **5.837 filas y 33 columnas**. Estas son las dimensiones de entrada, no el tamaño de la población final. La huella permite comprobar después que el archivo no cambió. La unidad se validará mediante la unicidad del código en F2.

### 5.1 Roles y decisiones previstas para las variables
Clasificaremos las 33 columnas según su significado. Leerlas como texto protege los identificadores, pero no convierte todas las variables en conceptualmente textuales.

En el diccionario fuente el código vigente aparece como `EstablecimientoCodigoVigente`; en el CSV se llama `EstablecimientoCodigo`. Documentamos esa correspondencia y usaremos el nombre real. El diccionario presenta definiciones intercambiadas de latitud y longitud; ambas columnas quedan fuera del estudio territorial, que no forma parte del alcance.

In [8]:
diccionario = pd.DataFrame(
[('EstablecimientoCodigoAntiguo',
  'Código anterior del establecimiento',
  'Identificador',
  'Excluir',
  'Código histórico; usaremos el código vigente.'),
 ('EstablecimientoCodigo',
  'Código vigente del establecimiento',
  'Identificador',
  'Conservar para trazabilidad',
  'Comprueba unicidad; no se interpreta como cantidad.'),
 ('EstablecimientoCodigoMadreAntiguo',
  'Código anterior del establecimiento madre',
  'Identificador',
  'Excluir',
  'La jerarquía institucional no forma parte de la pregunta.'),
 ('EstablecimientoCodigoMadreNuevo',
  'Código vigente del establecimiento madre',
  'Identificador',
  'Excluir',
  'La jerarquía institucional no forma parte de la pregunta.'),
 ('RegionCodigo',
  'Código de región',
  'Categoría nominal',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('RegionGlosa',
  'Nombre de región',
  'Categoría nominal',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('SeremiSaludCodigo_ServicioDeSaludCodigo',
  'Código de dependencia jerárquica',
  'Categoría nominal',
  'Excluir',
  'No se compararán servicios de salud o SEREMI.'),
 ('SeremiSaludGlosa_ServicioDeSaludGlosa',
  'Nombre de dependencia jerárquica',
  'Categoría nominal',
  'Excluir',
  'No se compararán servicios de salud o SEREMI.'),
 ('TipoPertenenciaEstabGlosa',
  'Pertenencia al SNSS',
  'Categoría nominal',
  'Conservar para análisis',
  'Dimensión administrativa; primera variable de la pregunta.'),
 ('TipoEstablecimientoGlosa',
  'Tipo de establecimiento',
  'Categoría nominal',
  'Excluir',
  'El análisis acordado compara pertenencia y complejidad, sin estratificar por tipo.'),
 ('AmbitoFuncionamiento',
  'Ámbito de funcionamiento',
  'Categoría nominal',
  'Excluir',
  'No se compararán ámbitos; se mantendrá la unidad definida por el código del registro.'),
 ('EstablecimientoGlosa',
  'Nombre del establecimiento',
  'Texto identificador',
  'Excluir',
  'El código permite trazabilidad sin incorporar nombres a la matriz.'),
 ('Certificacion',
  'Certificación administrativa',
  'Categoría nominal',
  'Excluir',
  'No se evalúan certificación ni calidad de atención.'),
 ('DependenciaAdministrativa',
  'Institución responsable de administrar',
  'Categoría nominal',
  'Excluir',
  'No equivale a pertenencia al SNSS; queda fuera del análisis bivariado.'),
 ('NivelAtencionEstabglosa',
  'Nivel de atención',
  'Categoría ordinal',
  'Excluir',
  'Nivel de atención y nivel de complejidad son conceptos distintos.'),
 ('ComunaCodigo',
  'Código de comuna',
  'Categoría nominal',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('ComunaGlosa',
  'Nombre de comuna',
  'Categoría nominal',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('TipoViaGlosa',
  'Tipo de vía de acceso',
  'Categoría nominal',
  'Excluir',
  'Dato de dirección sin función en la pregunta.'),
 ('NombreVia',
  'Nombre de la vía',
  'Texto',
  'Excluir',
  'Dato de dirección sin función en la pregunta.'),
 ('Numero',
  'Número de dirección',
  'Texto identificador',
  'Excluir',
  'Número de dirección, no medida cuantitativa.'),
 ('TelefonoMovil_TelefonoFijo',
  'Teléfono de contacto',
  'Texto identificador',
  'Excluir',
  'Dato de contacto sin función en la pregunta.'),
 ('FechaInicioFuncionamientoEstab',
  'Fecha de inicio de funcionamiento',
  'Fecha',
  'Excluir',
  'La antigüedad no forma parte del objetivo.'),
 ('TieneServicioUrgencia',
  'Disponibilidad de urgencia',
  'Categoría nominal',
  'Excluir',
  'Urgencias fuera del alcance.'),
 ('TipoUrgencia',
  'Tipo de servicio de urgencia',
  'Categoría nominal',
  'Excluir',
  'Urgencias fuera del alcance.'),
 ('ClasificacionTipoSapu',
  'Clasificación del SAPU',
  'Categoría nominal',
  'Excluir',
  'Urgencias fuera del alcance.'),
 ('Latitud',
  'Coordenada de latitud',
  'Numérica continua',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('Longitud',
  'Coordenada de longitud',
  'Numérica continua',
  'Excluir',
  'Análisis territorial fuera del alcance.'),
 ('TipoSistemaSaludGlosa',
  'Tipo de prestador en el sistema',
  'Categoría nominal',
  'Excluir',
  'Público/privado no es equivalente a pertenencia al SNSS.'),
 ('EstadoFuncionamiento',
  'Estado administrativo y asistencial',
  'Categoría nominal',
  'Usar como filtro',
  'Define vigencia; después no integra la matriz analítica.'),
 ('NivelComplejidadEstabGlosa',
  'Nivel de complejidad del establecimiento',
  'Categoría ordinal',
  'Conservar para análisis',
  'Dimensión asistencial; segunda variable de la pregunta; orden baja, mediana, alta.'),
 ('TipoAtencionEstabGlosa',
  'Modalidad de atención',
  'Categoría nominal',
  'Excluir',
  'No se compararán modalidades de atención.'),
 ('FechaIncorporacion',
  'Fecha de incorporación al registro',
  'Fecha',
  'Excluir',
  'No identifica por sí sola la fecha de descarga ni de apertura.'),
 ('FechaCierre',
  'Fecha de cierre del establecimiento',
  'Fecha',
  'Excluir',
  'La vigencia se determinará mediante EstadoFuncionamiento; vacío no implica error.')],
    columns=["variable", "significado", "rol", "decision", "justificacion"]
)
assert set(diccionario["variable"]) == set(datos_verificacion.columns)
display(diccionario)

,variable,significado,rol,decision,justificacion
0,EstablecimientoCodigoAntiguo,Código anterior del establecimiento,Identificador,Excluir,Código histórico; usaremos el código vigente.
1,EstablecimientoCodigo,Código vigente del establecimiento,Identificador,Conservar para trazabilidad,Comprueba unicidad; no se interpreta como cantidad.
2,EstablecimientoCodigoMadreAntiguo,Código anterior del establecimiento madre,Identificador,Excluir,La jerarquía institucional no forma parte de la pregunta.
3,EstablecimientoCodigoMadreNuevo,Código vigente del establecimiento madre,Identificador,Excluir,La jerarquía institucional no forma parte de la pregunta.
4,RegionCodigo,Código de región,Categoría nominal,Excluir,Análisis territorial fuera del alcance.
5,RegionGlosa,Nombre de región,Categoría nominal,Excluir,Análisis territorial fuera del alcance.
6,SeremiSaludCodigo_ServicioDeSaludCodigo,Código de dependencia jerárquica,Categoría nominal,Excluir,No se compararán servicios de salud o SEREMI.
7,SeremiSaludGlosa_ServicioDeSaludGlosa,Nombre de dependencia jerárquica,Categoría nominal,Excluir,No se compararán servicios de salud o SEREMI.
8,TipoPertenenciaEstabGlosa,Pertenencia al SNSS,Categoría nominal,Conservar para análisis,Dimensión administrativa; primera variable de la pregunta.
9,TipoEstablecimientoGlosa,Tipo de establecimiento,Categoría nominal,Excluir,"El análisis acordado compara pertenencia y complejidad, sin estratificar por tipo."


**Interpretación.** Dos variables responden directamente a la pregunta. El código y el estado tienen funciones de control. Las restantes se documentan y se excluyen por el alcance, no por afirmar que carezcan de utilidad para otros estudios. La complejidad tiene un orden conceptual, pero no se supondrán distancias iguales entre sus niveles.

### 5.2 Requisitos del dataset
Comprobaremos los mínimos de tamaño y diversidad. El archivo original puede satisfacerlos aunque la matriz final tenga menos columnas: la selección debe responder al problema definido. La matriz reducida conservará la información necesaria para este análisis bivariado.

In [9]:
criterios = pd.DataFrame({
    "criterio": ["Al menos 2.000 filas en origen", "Al menos 12 columnas en origen", "Variables de distinta naturaleza", "Variables de la pregunta disponibles"],
    "cumple": [len(datos_verificacion) >= 2000, datos_verificacion.shape[1] >= 12, diccionario["rol"].nunique() > 1,
               {d["columna"] for d in PROYECTO["dimensiones"]["principales"]}.issubset(datos_verificacion.columns)]
})
display(criterios)
assert criterios["cumple"].all()

,criterio,cumple
0,Al menos 2.000 filas en origen,True
1,Al menos 12 columnas en origen,True
2,Variables de distinta naturaleza,True
3,Variables de la pregunta disponibles,True


## 6. Control de versiones
Consultaremos si el proyecto ya tiene un repositorio propio y si registra commits. La ausencia de historial se informará como pendiente; no se atribuirán contribuciones a los integrantes sin evidencia. Cada integrante deberá registrar sus aportes reales al trabajar en el repositorio compartido.

In [10]:
def revisar_git(raiz):
    """Consulta el repositorio propio sin crear commits ni modificar Git."""
    if shutil.which("git") is None:
        return {"estado": "Git no disponible", "historial": "Pendiente"}
    if not (raiz / ".git").exists():
        return {"estado": "No hay repositorio propio en esta carpeta", "historial": "Pendiente de trabajo del grupo"}
    consulta = subprocess.run(["git", "log", "-5", "--format=%h %an: %s"], cwd=raiz, capture_output=True, text=True)
    return {"estado": "Repositorio detectado", "historial": consulta.stdout.strip() or "Sin commits verificables"}

estado_git = revisar_git(RAIZ)
display(pd.DataFrame([estado_git]))

,estado,historial
0,Repositorio detectado,f65dadd christopherulloarojas: Merge pull request #8 from christopherulloarojas/parte5-integracion\n037afef DeusExMachinaProject: integracion\ne14e643 BacmUnab: Merge pull request #7 from christopherulloarojas/parte3-limpieza\naf65afb BacmUnab: Docs\n37ef5ae kmartinezgimenez: Merge pull request #6 from christopherulloarojas/parte2-definicion


## 7. Validación técnica de F1
Comprobaremos la estructura, la definición y la cobertura del diccionario. Separaremos estas verificaciones técnicas de los pendientes documentales: un cuaderno ejecutable no acredita por sí solo la fecha de descarga, el mapa previo o la participación en Git.

In [11]:
def validar_fase1():
    """Valida los elementos necesarios para continuar con F2."""
    assert all((RAIZ / carpeta).is_dir() for carpeta in CARPETAS)
    assert len(PROYECTO["integrantes"]) == 4
    assert PROYECTO["grupo"] == "Grupo 1"
    assert all(PROYECTO[campo] for campo in ["pregunta", "objetivo_general", "objetivos_especificos"])
    assert set(diccionario["variable"]) == set(datos_verificacion.columns)
    assert [d["dimension"] for d in PROYECTO["dimensiones"]["principales"]] == ["Administrativa", "Asistencial"]
    assert (RAIZ / "requirements.txt").stat().st_size > 0
    assert (RAIZ / "src/utilidades.py").is_file()
    return "Validación técnica F1: correcta"

print(validar_fase1())

Validación técnica F1: correcta


## 8. Vinculación conceptual
Relacionaremos los conceptos del proyecto con sus evidencias. Esta tabla documenta la estructura conceptual de este trabajo.

In [12]:
vinculacion = pd.DataFrame([
    ["Problema y objetivos", "F1 §1", "PROYECTO"],
    ["Entorno y módulos", "F1 §2–4", "requirements.txt y src/utilidades.py"],
    ["Fuente y variables", "F1 §5", "Ficha y diccionario analítico"],
    ["Población y limpieza", "F2 §3", "Tabla de filtros y exclusiones"],
    ["Descripción y codificación", "F2 §4", "Gráficos, contingencia y matriz numérica"],
    ["Validación y comunicación", "F2 §6–8", "Comprobaciones y conclusiones"]
], columns=["concepto", "ubicacion", "evidencia"])
display(vinculacion)

,concepto,ubicacion,evidencia
0,Problema y objetivos,F1 §1,PROYECTO
1,Entorno y módulos,F1 §2–4,requirements.txt y src/utilidades.py
2,Fuente y variables,F1 §5,Ficha y diccionario analítico
3,Población y limpieza,F2 §3,Tabla de filtros y exclusiones
4,Descripción y codificación,F2 §4,"Gráficos, contingencia y matriz numérica"
5,Validación y comunicación,F2 §6–8,Comprobaciones y conclusiones


## 9. Persistencia y cierre
Guardaremos la configuración y las tablas para que F2 continúe desde la misma definición. Los metadatos distinguirán la procedencia declarada de las fechas no acreditadas.

In [13]:
(RAIZ / "docs/proyecto.json").write_text(json.dumps(PROYECTO, ensure_ascii=False, indent=2), encoding="utf-8")
diccionario.to_csv(RAIZ / "docs/diccionario_analitico.csv", index=False, encoding="utf-8-sig")
ficha.to_csv(RAIZ / "docs/ficha_dataset.csv", index=False, encoding="utf-8-sig")
vinculacion.to_csv(RAIZ / "docs/vinculacion_conceptual.csv", index=False, encoding="utf-8-sig")
metadatos = {"filas_originales": len(datos_verificacion), "columnas_originales": datos_verificacion.shape[1],
             "sha256": huella_original, "fuente": PROYECTO["fuente"], "versiones": versiones,
             "git": {"estado": "", "historial": ""}, "mapa_previo": ""}
(RAIZ / "docs/metadatos_f1.json").write_text(json.dumps(metadatos, ensure_ascii=False, indent=2), encoding="utf-8")
print("Configuración, ficha, diccionario y metadatos guardados.")

Configuración, ficha, diccionario y metadatos guardados.


**Conclusión de F1.** Se delimitó una pregunta bivariada sobre establecimientos vigentes con códigos únicos y categorías válidas. Se verificaron las 33 columnas del archivo, se documentó el papel de cada una y se prepararon funciones y rutas reproducibles. F2 deberá comprobar la población efectiva y describir sus distribuciones.

**Pendientes documentales:** acreditar fecha original de descarga y corte; contrastar esta estructura con el mapa conceptual previo; incorporar el historial real de trabajo del grupo en Git. Estos pendientes no se sustituyen por datos inventados.